# Early Cirrhosis Prediction — Feasibility Check (MIMIC-IV)

**Question:** among *at-risk* patients who are **not** cirrhotic at baseline, can routine labs
(FIB-4 / APRI) flag those who later receive a cirrhosis code — i.e. predict cirrhosis *earlier
than it is diagnosed*?

This notebook is a **feasibility check, not a model.** It answers four things:
- **(a)** how many at-risk patients exist without baseline cirrhosis,
- **(b)** how many of them later get a cirrhosis code (in a *later* admission),
- **(c)** how many have repeat labs spanning the window (needed for trajectory work),
- **(d)** whether **baseline FIB-4/APRI** separates future-cirrhosis from not (quick AUC).

## Honest framing (read first)
- "Earlier" here means **earlier than the diagnosis code**, not catching *early-stage* disease.
  FIB-4/APRI are sensitive to **advanced** fibrosis; early compensated cirrhosis has near-normal
  labs. So a positive result = *case-finding of not-yet-coded advanced disease*, not screening.
- MIMIC-IV has **limited longitudinal coverage** (most patients appear once). The binding
  question is whether enough at-risk patients have a *later* admission with a cirrhosis code —
  this notebook measures exactly that. A small (b)/(c) means the longitudinal study is not viable
  in MIMIC, regardless of signal.

## Definitions
- **At-risk:** baseline admission coded with chronic HCV, chronic HBV, alcohol use/alcoholic
  (non-cirrhotic) liver disease, NAFLD/NASH, or hemochromatosis.
- **Baseline:** each patient's earliest admission. Patients already coded cirrhotic at baseline
  are excluded.
- **Future cirrhosis (label=1):** a cirrhosis code in any admission *after* baseline.
- **FIB-4** = (age × AST) / (platelets × √ALT);  **APRI** = (AST / 40) / platelets × 100.

*Requires MIMIC-IV `hosp/`: diagnoses_icd, admissions, patients, labevents.*

## Config

In [1]:
# ============================================================
# CONFIG
# ============================================================
import os, json, warnings
import numpy as np
import pandas as pd
from pathlib import Path
warnings.filterwarnings("ignore")

CANDIDATE_ROOTS = ["/Users/sumiyasayeed/Documents/mimic4/mimiciv",
                   "../data/mimiciv", "./data/mimiciv", "../../data/mimiciv"]
MIMIC_ROOT = next((p for p in CANDIDATE_ROOTS if os.path.exists(os.path.join(p,"hosp"))),
                  CANDIDATE_ROOTS[0])
HOSP = os.path.join(MIMIC_ROOT, "hosp")
OUT = Path("results/feasibility_output"); OUT.mkdir(exist_ok=True)
CHUNK = 1_000_000
print("MIMIC_ROOT =", MIMIC_ROOT, "| hosp exists:", os.path.exists(HOSP))

def _csv(d,n):
    for e in (".csv",".csv.gz"):
        if os.path.exists(os.path.join(d,n+e)): return os.path.join(d,n+e)
    return os.path.join(d,n+".csv")

# ---- ICD prefixes (dots removed, upper) -------------------
RISK = {
  "HCV":            {9:("07044","07054","07070","07071"), 10:("B182","B1920","B1921")},
  "HBV":            {9:("07022","07023","07032","07033","07042","07052"), 10:("B180","B181","B1910","B1911")},
  "alcohol":        {9:("303","3050","5710","5711","5713"), 10:("F10","K700","K701","K709")},
  "NAFLD_NASH":     {9:("5718","5719"), 10:("K7581","K760","K7589")},
  "hemochromatosis":{9:("27501","27503"), 10:("E8310","E8311")},
}
CIRR = {9:("5712","5715","5716"),
        10:("K7030","K7031","K717","K740","K741","K742","K743","K744","K745","K746")}

# Lab itemids (hosp/labevents)
AST_ID, ALT_ID, PLT_ID = 50878, 50861, 51265
AST_ULN = 40.0
print("Config loaded.")


MIMIC_ROOT = /Users/sumiyasayeed/Documents/mimic4/mimiciv | hosp exists: True
Config loaded.


## Step 1 — diagnoses + admissions

In [2]:
# ============================================================
# STEP 1 — diagnoses + admissions -> per-admission risk/cirrhosis flags
# ============================================================
def _pref(code_up, ver_series, p9, p10):
    return ((ver_series==9)  & code_up.str.startswith(p9)) | \
           ((ver_series==10) & code_up.str.startswith(p10))

dx = pd.read_csv(_csv(HOSP,"diagnoses_icd"), usecols=["subject_id","hadm_id","icd_code","icd_version"])
code = dx["icd_code"].astype(str).str.replace(".","",regex=False).str.upper()
ver  = dx["icd_version"]
dx["is_cirr"] = _pref(code, ver, CIRR[9], CIRR[10]).astype(int)
for name, c in RISK.items():
    dx[name] = _pref(code, ver, c[9], c[10]).astype(int)
risk_cols = list(RISK)
perh = dx.groupby(["subject_id","hadm_id"])[["is_cirr"]+risk_cols].max().reset_index()

adm = pd.read_csv(_csv(HOSP,"admissions"), usecols=["subject_id","hadm_id","admittime"])
adm["admittime"] = pd.to_datetime(adm["admittime"], errors="coerce")
adm = adm.merge(perh, on=["subject_id","hadm_id"], how="left").fillna(0)
adm = adm.sort_values(["subject_id","admittime"])
print(f"✅ {len(adm):,} admissions across {adm.subject_id.nunique():,} patients")


✅ 546,028 admissions across 223,452 patients


## Step 2 — at-risk cohort + progression (a,b)

In [3]:
# ============================================================
# STEP 2 — build at-risk cohort + progression counts (a, b)
# ============================================================
base = adm.drop_duplicates("subject_id", keep="first").copy()           # earliest admission
base["any_risk"] = base[list(RISK)].max(axis=1)

# future cirrhosis = cirrhosis in a strictly later admission
base_time = dict(zip(base.subject_id, base.admittime))
adm["base_time"] = adm.subject_id.map(base_time)
later = adm[adm.admittime > adm.base_time]
future_cirr = later.groupby("subject_id")["is_cirr"].max()
base["future_cirr"] = base.subject_id.map(future_cirr).fillna(0).astype(int)
base["n_admissions"] = base.subject_id.map(adm.groupby("subject_id").size())

cohort = base[(base.any_risk==1) & (base.is_cirr==0)].copy()             # at-risk, not cirrhotic at baseline
print("="*60)
print("(a) at-risk patients without baseline cirrhosis : {:,}".format(len(cohort)))
print("(b) ... who later get a cirrhosis code          : {:,} ({:.1f}%)".format(
        int(cohort.future_cirr.sum()), cohort.future_cirr.mean()*100))
print("    (at-risk patients with >1 admission         : {:,})".format(int((cohort.n_admissions>1).sum())))
print("="*60)
print("Risk-factor breakdown (at baseline, non-cirrhotic):")
for r in RISK:
    sub = cohort[cohort[r]==1]
    print(f"  {r:<16} n={len(sub):<6} -> later cirrhosis: {int(sub.future_cirr.sum()):<5} ({sub.future_cirr.mean()*100:.1f}%)")


(a) at-risk patients without baseline cirrhosis : 20,841
(b) ... who later get a cirrhosis code          : 903 (4.3%)
    (at-risk patients with >1 admission         : 8,185)
Risk-factor breakdown (at baseline, non-cirrhotic):
  HCV              n=3306   -> later cirrhosis: 350   (10.6%)
  HBV              n=666    -> later cirrhosis: 52    (7.8%)
  alcohol          n=15174  -> later cirrhosis: 468   (3.1%)
  NAFLD_NASH       n=2624   -> later cirrhosis: 139   (5.3%)
  hemochromatosis  n=251    -> later cirrhosis: 18    (7.2%)


## Step 3 — baseline FIB-4/APRI + coverage (c)

In [4]:
# ============================================================
# STEP 3 — baseline labs -> FIB-4 / APRI  + longitudinal coverage (c)
# ============================================================
pts = pd.read_csv(_csv(HOSP,"patients"), usecols=["subject_id","anchor_age"])
age = dict(zip(pts.subject_id, pts.anchor_age))

base_hadm = dict(zip(cohort.subject_id, cohort.hadm_id))            # baseline hadm per cohort patient
base_hadm_set = set(cohort.hadm_id.astype(int))
# future cirrhosis hadm (first later admission with cirrhosis) for trajectory coverage
_cohort_subj = set(cohort.subject_id)
fut = (later[(later.is_cirr==1) & (later.subject_id.isin(_cohort_subj))]
       .sort_values("admittime").drop_duplicates("subject_id", keep="first"))  # cohort cases only
fut_hadm_set = set(fut.hadm_id.astype(int))

want_ids = {AST_ID, ALT_ID, PLT_ID}
def collect(hadm_set):
    acc = {}
    for ch in pd.read_csv(_csv(HOSP,"labevents"),
                          usecols=["hadm_id","itemid","valuenum"], chunksize=CHUNK, low_memory=False):
        ch = ch[ch.hadm_id.isin(hadm_set) & ch.itemid.isin(want_ids) & ch.valuenum.notna()]
        if ch.empty: continue
        for (h,i), v in ch.groupby(["hadm_id","itemid"])["valuenum"].median().items():
            acc[(int(h),int(i))] = v
    return acc

base_labs = collect(base_hadm_set)
def trip(hadm):
    return (base_labs.get((hadm,AST_ID)), base_labs.get((hadm,ALT_ID)), base_labs.get((hadm,PLT_ID)))
rows = []
for sid, hadm in base_hadm.items():
    ast_, alt_, plt_ = trip(int(hadm)); a = age.get(sid, np.nan)
    fib4 = (a*ast_)/(plt_*np.sqrt(alt_)) if all(pd.notna([a,ast_,alt_,plt_])) and alt_>0 and plt_>0 else np.nan
    apri = (ast_/AST_ULN)/plt_*100 if pd.notna(ast_) and pd.notna(plt_) and plt_>0 else np.nan
    rows.append(dict(subject_id=sid, base_ast=ast_, base_alt=alt_, base_plt=plt_, fib4=fib4, apri=apri))
labdf = pd.DataFrame(rows)
cohort = cohort.merge(labdf, on="subject_id", how="left")

# (c) longitudinal coverage
fut_labs = collect(fut_hadm_set)
fut_has_triple = {h for h in fut_hadm_set
                  if all((h,i) in fut_labs for i in want_ids)}
fut["has_fut_triple"] = fut.hadm_id.astype(int).isin(fut_has_triple)
n_base_fib4 = cohort["fib4"].notna().sum()
n_case_base_fib4 = cohort[cohort.future_cirr==1]["fib4"].notna().sum()
n_case_traj = fut[fut.has_fut_triple].subject_id.nunique()
print("(c) longitudinal lab coverage:")
print(f"    at-risk patients with computable BASELINE FIB-4 : {n_base_fib4:,}")
print(f"    future-cirrhosis cases with baseline FIB-4      : {n_case_base_fib4:,}")
print(f"    future-cirrhosis cases with labs at BOTH baseline & cirrhosis admission (trajectory): {n_case_traj:,}")
cohort.to_csv(OUT/"feasibility_cohort.csv", index=False)


(c) longitudinal lab coverage:
    at-risk patients with computable BASELINE FIB-4 : 8,235
    future-cirrhosis cases with baseline FIB-4      : 567
    future-cirrhosis cases with labs at BOTH baseline & cirrhosis admission (trajectory): 4,261


## Step 4 — separation AUC (d)

In [5]:
# ============================================================
# STEP 4 — does baseline FIB-4 / APRI separate future cirrhosis? (d)
# ============================================================
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

d = cohort.dropna(subset=["fib4"]).copy()
y = d["future_cirr"].values
print(f"Evaluable (baseline FIB-4 present): {len(d):,} | future-cirrhosis: {int(y.sum()):,} ({y.mean()*100:.1f}%)\n")

def boot_auc(yt, s, n=2000, seed=42):
    rng = np.random.default_rng(seed); a=[]
    yt=np.asarray(yt); s=np.asarray(s)
    for _ in range(n):
        idx=rng.integers(0,len(yt),len(yt))
        if not (1<yt[idx].sum()<len(idx)-1): continue
        a.append(roc_auc_score(yt[idx], s[idx]))
    return (np.median(a), np.percentile(a,2.5), np.percentile(a,97.5)) if a else (np.nan,)*3

if y.sum() >= 10 and len(np.unique(y))>1:
    for col in ["fib4","apri"]:
        m=d[col].notna().values
        med,lo,hi = boot_auc(y[m], d[col].values[m])
        print(f"  baseline {col.upper():<5} AUC = {med:.3f}  [95% CI {lo:.3f}-{hi:.3f}]")
    print("\n  FIB-4 by group (median [IQR]):")
    for g,lbl in [(0,"no future cirrhosis"),(1,"future cirrhosis")]:
        v=d[d.future_cirr==g]["fib4"]
        print(f"    {lbl:<22} n={len(v):<6} {v.median():.2f} [{v.quantile(.25):.2f}-{v.quantile(.75):.2f}]")
    # plot
    fig,ax=plt.subplots(1,2,figsize=(11,4))
    for g,c,lbl in [(0,"#4C72B0","no future cirrhosis"),(1,"#C44E52","future cirrhosis")]:
        ax[0].hist(np.clip(d[d.future_cirr==g]["fib4"],0,15), bins=30, alpha=.55, density=True, color=c, label=lbl)
    ax[0].axvline(1.45,ls="--",c="grey"); ax[0].axvline(3.25,ls="--",c="k")
    ax[0].set_xlabel("baseline FIB-4 (clipped at 15)"); ax[0].set_ylabel("density")
    ax[0].set_title("FIB-4 thresholds 1.45 / 3.25"); ax[0].legend(fontsize=8)
    d.boxplot(column="fib4", by="future_cirr", ax=ax[1]); ax[1].set_ylim(0,15)
    ax[1].set_title("baseline FIB-4 by future cirrhosis"); ax[1].set_xlabel("future cirrhosis"); plt.suptitle("")
    plt.tight_layout(); plt.savefig(OUT/"fib4_separation.png", dpi=150); plt.close()
    print("\n✅ Saved:", OUT/"fib4_separation.png", "and", OUT/"feasibility_cohort.csv")
else:
    print("  Too few future-cirrhosis cases to evaluate separation — the longitudinal cohort is not viable in this extract.")

print("\nVERDICT GUIDE:")
print("  Worth pursuing if (b) and (c) are in the hundreds AND baseline FIB-4 AUC clearly > 0.6.")
print("  If (b)/(c) are tiny, MIMIC lacks the longitudinal follow-up for an early-prediction study.")


Evaluable (baseline FIB-4 present): 8,235 | future-cirrhosis: 567 (6.9%)

  baseline FIB4  AUC = 0.696  [95% CI 0.674-0.717]
  baseline APRI  AUC = 0.662  [95% CI 0.640-0.684]

  FIB-4 by group (median [IQR]):
    no future cirrhosis    n=7668   1.67 [0.96-3.26]
    future cirrhosis       n=567    3.51 [1.90-6.24]

✅ Saved: feasibility_output/fib4_separation.png and feasibility_output/feasibility_cohort.csv

VERDICT GUIDE:
  Worth pursuing if (b) and (c) are in the hundreds AND baseline FIB-4 AUC clearly > 0.6.
  If (b)/(c) are tiny, MIMIC lacks the longitudinal follow-up for an early-prediction study.
